Testing calculations on punctuality and cancellations

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
data_selection = "french"

intermediate_outputs_dir = "intermediate_outputs"
data_path = f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_operators.parquet"

try: 
    data_chuuchuu.head()
except NameError:
    data_chuuchuu = pd.read_parquet(data_path)

In [3]:
cancelled_t = len(data_chuuchuu[data_chuuchuu["arrivalCancelled"]=="t"])/len(data_chuuchuu["arrivalCancelled"])*100
cancelled_f = len(data_chuuchuu[data_chuuchuu["arrivalCancelled"]=="f"])/len(data_chuuchuu["arrivalCancelled"])*100

print("Share of row arrivalCancelled = t")
print(round(cancelled_t,2),"%")
print("Share of row arrivalCancelled = f")
print(round(cancelled_f,2),"%")
print("Share of row without data cancelation")
print(round(100-(cancelled_f+cancelled_t),2),"%")

Share of row arrivalCancelled = t
0.36 %
Share of row arrivalCancelled = f
87.22 %
Share of row without data cancelation
12.42 %


### Recovering a cancellation status for rows where `arrivalCancelled` is null

Per the terminology doc: *"If null and a delay was recorded, you can assume the arrival was not cancelled."*

Two ways to check "was something recorded": whether `arrivalDelay` is non-null, or whether the effective `arrival` timestamp itself is non-null. These aren't the same thing -- `arrivalDelay` also needs `plannedArrival` to be present to be computable, so a row can have a real recorded `arrival` time with a still-`NaN` `arrivalDelay` (missing `plannedArrival`). Checking `arrival` directly is a strict superset of the delay-based check and matches the actual question ("did the train effectively arrive?") more directly, so that's the criterion used below.

In [4]:
null_cancelled = data_chuuchuu["arrivalCancelled"].isna()

# among the rows with no explicit flag, an effective arrival timestamp means it wasn't cancelled
inferred_not_cancelled = null_cancelled & data_chuuchuu["arrival"].notna()

# still no way to know -- neither an explicit flag nor an effective arrival to infer from
unreliable = null_cancelled & data_chuuchuu["arrival"].isna()

n = len(data_chuuchuu)
share_explicit = (~null_cancelled).sum() / n * 100
share_inferred = inferred_not_cancelled.sum() / n * 100
share_unreliable = unreliable.sum() / n * 100

print(f"Share of rows with an explicit arrivalCancelled value (t/f): {share_explicit:.2f}%")
print(f"Share of rows with arrivalCancelled null but inferred not cancelled (arrival is not null): {share_inferred:.2f}%")
print(f"Share of rows with no reliable cancellation data at all: {share_unreliable:.2f}%")
print()
print(f"Total reliable cancellation data: {share_explicit + share_inferred:.2f}%")

Share of rows with an explicit arrivalCancelled value (t/f): 87.58%
Share of rows with arrivalCancelled null but inferred not cancelled (arrival is not null): 0.00%
Share of rows with no reliable cancellation data at all: 12.42%

Total reliable cancellation data: 87.58%


In [5]:
data_chuuchuu["arrivalCancelled_resolved"] = data_chuuchuu["arrivalCancelled"]
data_chuuchuu.loc[inferred_not_cancelled, "arrivalCancelled_resolved"] = "f"

data_chuuchuu["arrivalCancelled_resolved"].value_counts(dropna=False)

arrivalCancelled_resolved
f      2524439
NaN     359480
t        10533
Name: count, dtype: int64

In [6]:
unreliable_rows = data_chuuchuu[unreliable]
unreliable_rows

,agency,routeType,routeNumber,date,deutscheBahnStopId,timestamp,originalRoute,originalStopId,stopName,arrival,...,sort_time_source,journey_id,is_ambiguous_trip,journey_verificator,is_cross_agency_duplicate,cross_agency_duplicate_confidence,depart_terminus,journey_type,normalized_operator,arrivalCancelled_resolved
6,FR,INTERCITES,3604,2025-01-01,8700036,2025-01-01 07:35:12.68+00,INTERCITES 3604,StopPoint:OCEINTERCITES-87594002,Brive-la-Gaillarde,NaT,...,departure,FR_INTERCITES_3604_2025-01-01,False,INTERCITES_3604_2025-01-01_8700036,False,not_flagged,depart,domestic,SNCF,NaN
8,FR,TGV INOUI,2501,2025-01-01,8700011,2025-01-01 07:44:48.176+00,TGV INOUI 2501,StopPoint:OCETGV INOUI-87113001,Paris Est,NaT,...,departure,FR_TGV INOUI_2501_2025-01-01,False,TGV INOUI_2501_2025-01-01_8700011,False,not_flagged,depart,domestic,SNCF,NaN
15,FR,INTERCITES,5950,2025-01-01,8700038,2025-01-01 07:55:25.245+00,INTERCITES 5950,StopPoint:OCEINTERCITES-87734004,Clermont-Ferrand,NaT,...,departure,FR_INTERCITES_5950_2025-01-01,False,INTERCITES_5950_2025-01-01_8700038,False,not_flagged,depart,domestic,SNCF,NaN
16,FR,INTERCITES,5954,2025-01-01,8700038,2025-01-01 08:05:23.364+00,INTERCITES 5954,StopPoint:OCEINTERCITES-87734004,Clermont-Ferrand,NaT,...,departure,FR_INTERCITES_5954_2025-01-01,False,INTERCITES_5954_2025-01-01_8700038,False,not_flagged,depart,domestic,SNCF,NaN
20,FR,TGV INOUI,8970,2025-01-01,8700200,2025-01-01 08:14:31.097+00,TGV INOUI 8970,StopPoint:OCETGV INOUI-87486449,Les Sables-d'Olonne,NaT,...,departure,FR_TGV INOUI_8970_2025-01-01,False,TGV INOUI_8970_2025-01-01_8700200,False,not_flagged,depart,domestic,SNCF,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2894423,FR,TGV INOUI,8454,2025-12-31,8700047,2026-01-01 00:07:22.222+00,TGV INOUI 8454,StopPoint:OCETGV INOUI-87581009,Bordeaux Saint-Jean,NaT,...,departure,FR_TGV INOUI_8454_2025-12-31,False,TGV INOUI_8454_2025-12-31_8700047,False,not_flagged,depart,domestic,SNCF,NaN
2894428,FR,CAR TER,48271,2025-12-31,8700015,2026-01-01 00:07:23.085+00,Car TER 48271,StopPoint:OCECar TER-87384008,Paris Saint-Lazare,NaT,...,departure,FR_CAR TER_48271_2025-12-31,False,CAR TER_48271_2025-12-31_8700015,False,not_flagged,depart,domestic,NaN,NaN
2894435,FR,CAR TER,34519,2025-12-31,8700068,2026-01-01 00:07:24.887+00,Car TER 34519,StopPoint:OCECar TER-87745000,Bellegarde,NaT,...,departure,FR_CAR TER_34519_2025-12-31,False,CAR TER_34519_2025-12-31_8700068,False,not_flagged,depart,domestic,NaN,NaN
2894440,FR,TRAIN TER,860569,2025-12-31,8700010,2026-01-01 00:37:23.732+00,Train TER 860569,StopPoint:OCETrain TER-87547000,Paris Austerlitz,NaT,...,departure,FR_TRAIN TER_860569_2025-12-31,False,TRAIN TER_860569_2025-12-31_8700010,False,not_flagged,depart,domestic,SNCF,NaN


### Checking the literal terminology-doc criterion: "if null and a delay was recorded, assume not cancelled"

This applies the rule exactly as written (`arrivalDelay` non-null), rather than the `arrival`-based criterion used above, so the two can be compared directly.

In [7]:
inferred_not_cancelled_delay = null_cancelled & data_chuuchuu["arrivalDelay"].notna()
unreliable_delay = null_cancelled & data_chuuchuu["arrivalDelay"].isna()

share_inferred_delay = inferred_not_cancelled_delay.sum() / n * 100
share_unreliable_delay = unreliable_delay.sum() / n * 100

print(f"Share of rows with arrivalCancelled null but inferred not cancelled (arrivalDelay is not null): {share_inferred_delay:.2f}%")
print(f"Share of rows with no reliable cancellation data at all (delay-based criterion): {share_unreliable_delay:.2f}%")
print()
print(f"Total reliable cancellation data (delay-based criterion): {share_explicit + share_inferred_delay:.2f}%")

Share of rows with arrivalCancelled null but inferred not cancelled (arrivalDelay is not null): 0.00%
Share of rows with no reliable cancellation data at all (delay-based criterion): 12.42%

Total reliable cancellation data (delay-based criterion): 87.58%


In [8]:
# rows where the arrival-based and delay-based criteria disagree (both only defined among null_cancelled rows)
criteria_disagree = inferred_not_cancelled != inferred_not_cancelled_delay
print(f"{criteria_disagree.sum()} rows where the two criteria disagree")

# by construction (arrivalDelay needs plannedArrival + arrival to both be present), the delay-based
# criterion can only be a subset of the arrival-based one -- confirm that's actually the case here
only_arrival_based = inferred_not_cancelled & ~inferred_not_cancelled_delay
only_delay_based = inferred_not_cancelled_delay & ~inferred_not_cancelled
print(f"rows caught by arrival-based but not delay-based: {only_arrival_based.sum()}")
print(f"rows caught by delay-based but not arrival-based: {only_delay_based.sum()} (expected to be 0)")

data_chuuchuu[only_arrival_based][["arrival", "plannedArrival", "arrivalDelay", "arrivalCancelled"]].head(20)

0 rows where the two criteria disagree
rows caught by arrival-based but not delay-based: 0
rows caught by delay-based but not arrival-based: 0 (expected to be 0)


,arrival,plannedArrival,arrivalDelay,arrivalCancelled


### Country ranking: domestic punctuality at the terminus (5-minute threshold)

Restricted to `journey_type == "domestic"` and `depart_terminus == "terminus"` -- the last stop of each resolved trip, so each row here is one journey's actual outcome rather than a per-stop measurement.

A terminus row is **assessable** only if its cancellation status is reliable (`arrivalCancelled_resolved` is not null) and either it was cancelled (automatically counted as not punctual) or its `arrivalDelay` is known. Rows failing that -- unreliable cancellation status, or not cancelled but with an unknown delay (the residual edge case identified above) -- are excluded rather than guessed at.

**Punctual** = not cancelled and `arrivalDelay <= 5` (minutes).

Countries with very few `n_terminus_arrivals` should be read with caution -- their rate is based on too few journeys to be meaningful.

In [9]:
scope = (data_chuuchuu["journey_type"] == "domestic") & (data_chuuchuu["depart_terminus"] == "terminus")

reliable_cancellation = data_chuuchuu["arrivalCancelled_resolved"].notna()
is_cancelled = data_chuuchuu["arrivalCancelled_resolved"] == "t"
has_delay = data_chuuchuu["arrivalDelay"].notna()

# assessable for punctuality: cancellation status is known, and either it was cancelled (automatically
# not punctual) or we have an actual delay to check against the 5-minute threshold
assessable = reliable_cancellation & (is_cancelled | has_delay)
punctual_5min = assessable & ~is_cancelled & (data_chuuchuu["arrivalDelay"] <= 300)#5 min = 300 seconds

mask = scope & assessable
country_punctuality = pd.DataFrame({
    "country": data_chuuchuu.loc[mask, "country"],
    "punctual": punctual_5min[mask],
})

country_ranking = country_punctuality.groupby("country").agg(
    n_terminus_arrivals=("punctual", "size"),
    n_punctual=("punctual", "sum"),
)
country_ranking["punctuality_rate_pct"] = (country_ranking["n_punctual"] / country_ranking["n_terminus_arrivals"] * 100).round(2)
country_ranking = country_ranking.sort_values("punctuality_rate_pct", ascending=False)

print(f"{(scope & ~assessable).sum()} domestic terminus rows excluded as not assessable (unreliable cancellation status or unknown delay)")
print()
country_ranking

11 domestic terminus rows excluded as not assessable (unreliable cancellation status or unknown delay)



,n_terminus_arrivals,n_punctual,punctuality_rate_pct
country,,,
Germany,10,10,100.00
Luxembourg,1,1,100.00
Italy,2,2,100.00
France,319529,277408,86.82
Belgium,1,0,0.00


### Building a per-train summary for the operator / route-type report (nb5)

Everything so far treats each row as one stop. The nb5 summary (by year, operator and route type) needs one row per train instead, so we collapse each `journey_id` down to a single record.

We reuse the same exclusions as nb2/nb3: a journey with a repeated stop id (`is_ambiguous_trip`) can't be ranked into depart/intermediate/terminus, and a `journey_verificator` collision flagged `likely_true_duplicate` is almost certainly the same physical train reported twice by two agencies -- keeping both would double-count it.

**Cancellation** is judged across every stop of the journey (using `arrivalCancelled_resolved` from above):
- a journey is only classified if every one of its stops has a known cancellation status -- a single unresolved stop could be hiding either outcome, so we don't guess
- **full cancellation**: all stops cancelled
- **partial cancellation**: some, but not all, stops cancelled

**Terminus outcome** (`cancelled_terminus`) looks specifically at the journey's `terminus` stop: was it cancelled, not cancelled, or unresolved (no terminus row could be identified for that journey)? `arrived_at_terminus` is true only when the terminus was resolved and confirmed not cancelled. Delay (>5min / >15min) is measured on that same terminus row's `arrivalDelay`, in line with the country-punctuality check above -- a cancelled or unresolved terminus has no delay to measure, so it's excluded from both delay counts rather than counted as on-time.

In [10]:
# nb5's operator/route-type summary needs one row per train, not one row per stop -- exclude the same
# unreliable journeys nb2/nb3 already flagged (see markdown above)
clean = ~data_chuuchuu["is_ambiguous_trip"] & (data_chuuchuu["cross_agency_duplicate_confidence"] != "likely_true_duplicate")
print(f"{(~clean).sum()} rows excluded as an ambiguous trip or a likely cross-agency duplicate")

clean_data = data_chuuchuu[clean].copy()
clean_data["year"] = pd.to_datetime(clean_data["date"]).dt.year

0 rows excluded as an ambiguous trip or a likely cross-agency duplicate


In [11]:
stop_counts = clean_data.groupby("journey_id").agg(
    n_stops=("arrivalCancelled_resolved", "size"),
    n_known_cancellation=("arrivalCancelled_resolved", lambda s: s.notna().sum()),
    n_cancelled_stops=("arrivalCancelled_resolved", lambda s: (s == "t").sum()),
)

# only classify a journey's cancellation if every one of its stops has a known status
stop_counts["cancellation_reliable"] = stop_counts["n_known_cancellation"] == stop_counts["n_stops"]
stop_counts["is_fully_cancelled"] = stop_counts["cancellation_reliable"] & (stop_counts["n_cancelled_stops"] == stop_counts["n_stops"])
stop_counts["is_partially_cancelled"] = (
    stop_counts["cancellation_reliable"]
    & (stop_counts["n_cancelled_stops"] > 0)
    & (stop_counts["n_cancelled_stops"] < stop_counts["n_stops"])
)

print(f"{(~stop_counts['cancellation_reliable']).sum()} of {len(stop_counts)} journeys have no reliable cancellation status (excluded from full/partial counts)")
stop_counts[["is_fully_cancelled", "is_partially_cancelled"]].sum()

357840 of 405471 journeys have no reliable cancellation status (excluded from full/partial counts)


is_fully_cancelled        1775
is_partially_cancelled    2646
dtype: int64

In [12]:
# journey attributes (operator, route type, year) are constant across a journey's stops by
# construction (journey_id == agency + routeType + routeNumber + date), so any stop can represent them
journey_attrs = clean_data.drop_duplicates("journey_id").set_index("journey_id")[
    ["agency", "normalized_operator", "routeType", "year", "journey_type"]
].rename(columns={"normalized_operator": "operator"})

terminus_rows = clean_data.loc[clean_data["depart_terminus"] == "terminus"].set_index("journey_id")
cancelled_terminus = terminus_rows["arrivalCancelled_resolved"].map({"t": True, "f": False})  # NaN where unresolved

journeys = journey_attrs.join(stop_counts)
journeys["cancelled_terminus"] = journeys.index.map(cancelled_terminus)
journeys["terminus_arrival_delay"] = journeys.index.map(terminus_rows["arrivalDelay"])

# arrived at terminus: the terminus stop was identified AND confirmed not cancelled (NaN --
# no terminus row could be resolved for that journey -- correctly evaluates to False here)
journeys["arrived_at_terminus"] = journeys["cancelled_terminus"] == False

DELAY_5MIN_SECONDS = 5 * 60
DELAY_15MIN_SECONDS = 15 * 60
journeys["delayed_5min"] = journeys["arrived_at_terminus"] & (journeys["terminus_arrival_delay"] > DELAY_5MIN_SECONDS)
journeys["delayed_15min"] = journeys["arrived_at_terminus"] & (journeys["terminus_arrival_delay"] > DELAY_15MIN_SECONDS)

print(f"{len(journeys)} journeys total")
print(f"{journeys['arrived_at_terminus'].sum()} arrived at terminus")
print(f"{journeys['delayed_5min'].sum()} delayed >5min, {journeys['delayed_15min'].sum()} delayed >15min (at terminus)")

journeys.head()

405471 journeys total
403169 arrived at terminus
61567 delayed >5min, 33604 delayed >15min (at terminus)


,agency,operator,routeType,year,journey_type,n_stops,n_known_cancellation,n_cancelled_stops,cancellation_reliable,is_fully_cancelled,is_partially_cancelled,cancelled_terminus,terminus_arrival_delay,arrived_at_terminus,delayed_5min,delayed_15min
journey_id,,,,,,,,,,,,,,,,
FR_INTERCITES_3604_2025-01-01,FR,SNCF,INTERCITES,2025,domestic,7,6,0,False,False,False,False,0.0,True,False,False
FR_TGV INOUI_2501_2025-01-01,FR,SNCF,TGV INOUI,2025,domestic,3,2,0,False,False,False,False,0.0,True,False,False
FR_INTERCITES_5950_2025-01-01,FR,SNCF,INTERCITES,2025,domestic,6,5,0,False,False,False,False,0.0,True,False,False
FR_INTERCITES_5954_2025-01-01,FR,SNCF,INTERCITES,2025,domestic,2,1,0,False,False,False,False,0.0,True,False,False
FR_TGV INOUI_8970_2025-01-01,FR,SNCF,TGV INOUI,2025,domestic,5,4,0,False,False,False,False,0.0,True,False,False


Exporting the per-train table for nb5

In [13]:
export_journeys = input("Export journey-level data to parquet? (y/n): ")

if export_journeys.lower() == "y":
    os.makedirs(intermediate_outputs_dir, exist_ok=True)

    journeys_export_path = f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_journeys.parquet"
    journeys.reset_index().to_parquet(journeys_export_path)
    print(f"Saved to {journeys_export_path}")

Saved to intermediate_outputs/data_chuuchuu_french_journeys.parquet
